In [1]:
import numpy as np

from decomps.lup import lup_factor
from metrics import condition_1_estimate, relative_error
from solvers.lup_solve import lup_solve

ALPHA_POWERS = [0, 10, 20, 30, 40]  # alpha = 2^-k
THETA = 2.0**-30
WILKINSON_SIZES = [8, 16, 32, 48, 64]

Обусловленность: направление возмущения

$$A_\alpha = \frac12\begin{bmatrix}1+\alpha & 1-\alpha\\ 1-\alpha & 1+\alpha\end{bmatrix},\qquad u = \begin{bmatrix}1\\1\end{bmatrix},\qquad v = \begin{bmatrix}1\\-1\end{bmatrix},\qquad b = u$$

$$b_+ = b + \theta u,\qquad b_- = b + \theta v,\qquad \theta = 2^{-30}$$

In [2]:
u = np.array([1.0, 1.0])
v = np.array([1.0, -1.0])


def make_A(alpha):
    return 0.5 * np.array([[1 + alpha, 1 - alpha],
                           [1 - alpha, 1 + alpha]])


def exact_solutions(alpha, theta):
    x_star = u.copy()                    # A u = u = b
    x_plus = u + theta * u               # A x+ = u + theta u
    x_minus = u + (theta / alpha) * v    # A x- = u + theta v
    return x_star, x_plus, x_minus

1. Точные решения

In [3]:
print(f"{'k':>3} {'A u':>14} {'A v / alpha':>14} "
      f"{'||x+ - x*||/||x*||':>20} {'||x- - x*||/||x*||':>20}")
for k in ALPHA_POWERS:
    alpha = 2.0**-k
    A = make_A(alpha)
    x_star, x_plus, x_minus = exact_solutions(alpha, THETA)
    print(f"{k:>3} {str(A @ u):>14} {str(A @ v / alpha):>14} "
          f"{relative_error(x_plus, x_star):>20.6e} "
          f"{relative_error(x_minus, x_star):>20.6e}")

  k            A u    A v / alpha   ||x+ - x*||/||x*||   ||x- - x*||/||x*||
  0        [1. 1.]      [ 1. -1.]         9.313226e-10         9.313226e-10
 10        [1. 1.]      [ 1. -1.]         9.313226e-10         9.536743e-07
 20        [1. 1.]      [ 1. -1.]         9.313226e-10         9.765625e-04
 30        [1. 1.]      [ 1. -1.]         9.313226e-10         1.000000e+00
 40        [1. 1.]      [ 1. -1.]         9.313226e-10         1.024000e+03


Сумма каждой строки $A_\alpha$ равна $\frac12\big((1+\alpha)+(1-\alpha)\big) = 1$, а разность элементов строки равна $\frac12\big((1+\alpha)-(1-\alpha)\big) = \alpha$. Поэтому

$$A_\alpha u = u,\qquad A_\alpha v = \alpha v,$$

то есть $u$ и $v$ -- собственные векторы $A_\alpha$ с собственными числами $1$ и $\alpha$.

Отсюда точные решения:

* $b = u \ \Rightarrow\  x^* = u$,
* $b_+ = (1+\theta)u \ \Rightarrow\  x_+ = (1+\theta)u$,
* $b_- = u + \theta v$. Ищем $x_- = u + cv$: $A(u + cv) = u + c\alpha v = u + \theta v$, откуда $c = \theta/\alpha$ и $x_- = u + \frac{\theta}{\alpha}v$.

Так как $\|u\|_2 = \|v\|_2 = \sqrt2$:

$$\frac{\|x_+ - x^*\|_2}{\|x^*\|_2} = \theta,\qquad \frac{\|x_- - x^*\|_2}{\|x^*\|_2} = \frac{\theta}{\alpha}.$$

Компоненты $x_-$ равны $1 \pm 2^{k-30}$ и точно представимы в `float64`, поэтому значения в таблице вычислены без округления.

Решение равно $A^{-1}b$, а $A^{-1}$ растягивает разные направления по-разному: вдоль $u$ в $1$ раз, вдоль $v$ в $1/\alpha$ раз. Возмущение вдоль $u$ не усиливается, а вдоль $v$ усиливается в $1/\alpha$ раз. При $\alpha = 2^{-40}$ относительное изменение решения равно $2^{10} = 1024$, хотя правая часть изменилась лишь на $\theta \approx 9.3\cdot10^{-10}$. То есть важно направление изменения

2. Число обусловленности

In [4]:
print(f"{'k':>3} {'1/alpha':>12} {'kappa~_1 (LUP)':>16} {'rel. diff':>12}")
for k in ALPHA_POWERS:
    alpha = 2.0**-k
    A = make_A(alpha)
    L, U, p = lup_factor(A)
    kappa_est = condition_1_estimate(A, lambda b: lup_solve(L, U, p, b))
    exact = 1 / alpha
    print(f"{k:>3} {exact:>12.6e} {kappa_est:>16.6e} "
          f"{abs(kappa_est - exact) / exact:>12.2e}")

  k      1/alpha   kappa~_1 (LUP)    rel. diff
  0 1.000000e+00     1.000000e+00     0.00e+00
 10 1.024000e+03     1.024000e+03     8.88e-16
 20 1.048576e+06     1.048576e+06     9.09e-13
 30 1.073742e+09     1.073742e+09     9.31e-10
 40 1.099512e+12     1.099512e+12     9.09e-13


**$\kappa_2$.** Разложим $z = au + cv$. Векторы $u, v$ ортогональны, поэтому

$$\|z\|_2^2 = 2(a^2 + c^2),\qquad Az = au + \alpha c\,v,\qquad \|Az\|_2^2 = 2(a^2 + \alpha^2c^2).$$

Отношение $\|Az\|_2/\|z\|_2$ максимально при $c = 0$ (равно $1$) и минимально при $a = 0$ (равно $\alpha$), значит

$$\kappa_2(A_\alpha) = \frac{\max_{\|z\|=1}\|Az\|_2}{\min_{\|z\|=1}\|Az\|_2} = \frac1\alpha.$$

**$\kappa_1$.** У $A_\alpha^{-1}$ те же собственные векторы с числами $1$ и $1/\alpha$:

$$A_\alpha^{-1} = \frac12\begin{bmatrix}1+\frac1\alpha & 1-\frac1\alpha\\[2pt] 1-\frac1\alpha & 1+\frac1\alpha\end{bmatrix}.$$

$\|\cdot\|_1$ -- максимальная сумма модулей по столбцам. При $0 < \alpha \le 1$:

$$\|A_\alpha\|_1 = \tfrac12\big(1+\alpha + 1-\alpha\big) = 1,\qquad \|A_\alpha^{-1}\|_1 = \tfrac12\Big(\tfrac1\alpha + 1 + \tfrac1\alpha - 1\Big) = \frac1\alpha,$$

$$\kappa_1(A_\alpha) = \kappa_2(A_\alpha) = \frac1\alpha.$$

**Оценка $\widetilde\kappa_1$ по LUP-факторам.** Раскладываем $A_\alpha$ один раз, решаем $A z_j = e_j$ векторы $z_j$ -- столбцы $A^{-1}$, и $\widetilde\kappa_1 = \|A\|_1\max_j\|z_j\|_1$.

Оценка совпадает с $1/\alpha$ с относительным расхождением не больше $\sim10^{-9}$. Это численная оценка: столбцы $A^{-1}$ сами вычислены с ошибкой порядка $\mathfrak u\,\kappa$, что при $\kappa \sim 10^9$ даёт $\sim10^{-7}$. Наблюдаемое расхождение даже меньше этого ориентира.